# 03 - Ollama Enrichment

This notebook enriches AI/ML jobs with role family, seniority, and skills using local Ollama.

## Load AI Candidates

In [1]:
from __future__ import annotations

import polars as pl

from job_market_copilot.config import get_settings
from job_market_copilot.processing.enrich import JobEnricher

settings = get_settings()
ai_candidates = pl.read_parquet(settings.resolved_artifacts_dir / 'jobs_ai_candidates.parquet')
ai_candidates.shape

(40, 16)

## Run Enrichment
The model is configurable through `OLLAMA_MODEL`. Defaults to `granite4.1:3b`.

In [2]:
enricher = JobEnricher(settings)
enriched_df = enricher.enrich_dataframe(ai_candidates)
enriched_df.write_parquet(settings.resolved_artifacts_dir / 'ai_jobs_enriched.parquet')
print(enriched_df.shape)
enriched_df.select(['title','role_family','seniority','core_skills','model_source']).head(20)

2026-06-12 12:08:02.480 | INFO     | job_market_copilot.processing.enrich:enrich_dataframe:87 - Enrichment progress: 10/12


2026-06-12 12:08:22.519 | INFO     | job_market_copilot.processing.enrich:enrich_dataframe:87 - Enrichment progress: 12/12


(40, 23)


title,role_family,seniority,core_skills,model_source
str,str,str,list[str],str
"""Business Transformation Lead""","""other""","""senior""","[""ai integration"", ""business transformation"", … ""executive partnership""]","""ollama"""
"""Mid/Senior AI Cinematic Video …","""other""","""unknown""",[],"""ollama"""
"""Customer Operations & Writing …","""other""","""unknown""",[],"""ollama"""
"""Office Assistant""","""other""","""intern""",[],"""ollama"""
"""Staff Product Engineer (Belo H…","""software engineer""","""senior""","[""backend"", ""frontend"", … ""research""]","""ollama"""
…,…,…,…,…
"""Staff Software Engineer, Produ…","""other""","""unknown""",[],"""fallback_limit"""
"""Tech Lead Full-Stack Rails Eng…","""other""","""unknown""",[],"""fallback_limit"""
"""Full Stack Web Developer ⏤ Rem…","""other""","""unknown""",[],"""fallback_limit"""


## Enrichment Source Quality

In [3]:
enriched_df.group_by('model_source').len(name='count').sort('count', descending=True)

model_source,count
str,u32
"""fallback_limit""",28
"""ollama""",12
